# DINÁMICA DE KAWASAKI

## Aspectos de simulación y entorno de cómputo

Para llevar a cabo la simulación numérica del modelo de Ising bidimensional con dinámica de Kawasaki se ha trabajado en un entorno de computación de altas prestaciones (HPC), ejecutando el código en el supercomputador **Proteus**.Este sistema opera bajo un entorno de tipo **Linux/Unix**, sobre el que se ha configurado un entorno virtual de Python aislado con la versión **Python 3.10.12**, lo que garantiza la reproducibilidad de la ejecución y un control preciso de las dependencias del proyecto. La simulación completa —que abarca redes de hasta 128×128 espines, múltiples temperaturas y varias realizaciones independientes para ambos valores de la magnetización inicial— se ha resuelto en un tiempo total de cómputo de aproximadamente **dos horas**.

En el plano del desarrollo, se ha utilizado inteligencia artificial generativa como herramienta de apoyo a la programación y a la estructuración del trabajo, actuando como un **motor de metaprogramación** que ayuda a transformar el planteamiento físico del problema en una implementación computacional eficiente. El flujo de trabajo ha seguido dos fases complementarias: en primer lugar se ha empleado el modelo Gemini para generar un boceto inicial del código, que posteriormente se ha ido perfeccionando; y, en una segunda fase, se ha recurrido a Claude para el análisis de los errores e ineficiencias de esa primera versión, que ha propuesto mejoras sucesivas hasta alcanzar la implementación definitiva.

A nivel conceptual, la IA ha actuado como un **orquestador semántico**, interpretando el problema físico y guiando la elección de los métodos de simulación más apropiados. El núcleo de la simulación es el **algoritmo de Metropolis aplicado a la dinámica de Kawasaki**, en la que el movimiento elemental de Monte Carlo consiste en el intercambio de un par de espines vecinos y no en el cambio de un único espín, de manera que la magnetización total del sistema se conserva a lo largo de toda la evolución. Sobre esta base se han incorporado varias técnicas propias de la simulación estadística para garantizar resultados fiables: la **inicialización del sistema en una configuración con separación de fases** (en lugar de una configuración aleatoria), un esquema de **calentamiento progresivo (*annealing*)** en el que la red equilibrada a una temperatura se emplea como punto de partida para la siguiente, evitando reinicializaciones que dejarían al sistema fuera del equilibrio a temperaturas altas; el **promedio sobre varias realizaciones independientes** con distintas semillas del generador de números aleatorios, para reducir el ruido estadístico; y un análisis de **escalado de tamaño finito (*finite-size scaling*)** que permite extrapolar la temperatura crítica al límite termodinámico (1/N → 0).

En la capa de ejecución, el aspecto computacional más relevante es el uso de la librería **Numba**, un compilador *just-in-time* (JIT) que traduce las funciones críticas de Python a código máquina optimizado a través de la infraestructura de compilación **LLVM**. Mediante el decorador `@njit`, los bucles internos de Monte Carlo —que implican del orden de 10⁹ intentos de intercambio de espines por simulación— se ejecutan sin la sobrecarga del intérprete de Python, lo que reduce drásticamente el tiempo de cómputo y hace viable el muestreo de las redes de mayor tamaño. Antes de cada ejecución se configuraron además las variables de entorno relativas al uso de hilos, fijando a un único hilo las bibliotecas de álgebra lineal subyacentes (BLAS, MKL y OpenMP) con el fin de evitar la sobre-suscripción de recursos en un nodo compartido y mantener un uso controlado y reproducible de la máquina. No obstante, conviene precisar que la aceleración del cálculo proviene fundamentalmente de la compilación JIT y **no de una paralelización explícita** del código, de modo que los bucles de Monte Carlo se ejecutan como código compilado esencialmente en serie. El resto de la implementación se apoya en las librerías estándar del ecosistema científico de Python: **NumPy** para el cálculo numérico y el manejo eficiente de las matrices de espines, y **Matplotlib** para la representación gráfica de las magnitudes termodinámicas y de las configuraciones instantáneas del sistema.

En conjunto, el uso de inteligencia artificial generativa no sustituye en ningún caso el análisis físico ni la comprensión de los métodos de Monte Carlo empleados, sino que introduce una capa de asistencia y metaprogramación que permite organizar el desarrollo, reducir errores de implementación, optimizar el rendimiento del código y conectar de forma más directa la formulación estadística del modelo de Ising con su implementación computacional sobre el entorno de cálculo de Proteus.

## INTRODUCCIÓN :  

## El Modelo de Ising y la Dinámica de Kawasaki
 
El modelo de Ising constituye uno de los marcos teóricos más célebres para el estudio del ferromagnetismo y las transiciones de fase, basándose en una red de espines que interactúan con sus vecinos más próximos. Mientras que la dinámica convencional de Glauber permite que estos espines cambien su estado de forma individual —comportándose como un sistema que intercambia energía y magnetización con un reservorio externo—, el modelo de Ising con parámetro de orden conservado (COP) impone una restricción fundamental: el número total de espines en cada estado debe permanecer constante. Esta restricción se implementa mediante la dinámica de Kawasaki, un algoritmo de Monte Carlo que, en lugar de voltear espines, propone el intercambio de valores entre pares vecinos. Esta regla convierte efectivamente al sistema en un modelo de gas de red, donde las partículas se difunden y se segregan en dominios o "gotas" al enfriarse, reflejando de manera cualitativa el comportamiento físico real de la separación de fases en fluidos y aleaciones.

### Diferencias Fundamentales: Dinámica de Kawasaki vs. Glauber

La distinción esencial entre el modelo estándar de Ising y el modelo COP reside en su dinámica y las restricciones impuestas sobre el sistema:

* Conservación de la Magnetización: A diferencia del modelo de Ising clásico, donde la magnetización total $M = \sum s_i$ puede variar, en el modelo COP la magnetización es una invariante del movimiento. Físicamente, esto transforma el sistema de un imán a un gas de red, donde los espines $+1$ y $-1$ representan la presencia o ausencia de partículas que se difunden por la red, manteniendo una densidad constante $\rho$.  

* Mecanismo de Evolución: Mientras que la dinámica de Glauber recurre al spin-flip (volteo individual de espines), la dinámica de Kawasaki implementa el intercambio local de un par de espines vecinos. Este movimiento preserva el número total de partículas y permite el estudio de procesos de difusión y separación de fases, tales como la formación de dominios o gotas de una fase condensada. 

## Objetivos de la Simulación

El objetivo principal de este trabajo es implementar un algoritmo de Monte Carlo basado en el criterio de Metropolis para estudiar el comportamiento térmico del modelo COP en dos dimensiones. Analizaremos cómo, bajo una magnetización inicial nula, el sistema experimenta una transición de fase continua hacia un estado de coexistencia de dominios. Asimismo, exploraremos el régimen de magnetización no nula, donde la dinámica induce una transición discontinua característica de una separación de fases de primer orden.

Para ello, compararemos los resultados obtenidos mediante el cálculo de magnitudes termodinámicas clave —como la magnetización por dominio, la energía media, el calor específico y la susceptibilidad magnética— en redes de distintos tamaños ($N=32, 64, 128, 256, 512, 1028$), permitiéndonos extraer conclusiones sobre el comportamiento crítico en el límite termodinámico ($N \to \infty$)

# Metodología : 

## Implementación del Modelo COP

El núcleo de nuestra simulación se basa en la dinámica de Kawasaki , la cual preserva la magnetización total $M = N(2\rho - 1)$. A diferencia de los algoritmos de spin-flip, el movimiento elemental consiste en el intercambio de espines en sitios vecinos aleatorios.  Para garantizar la eficiencia computacional, no recalculamos la energía total $E(C)$ en cada paso, sino que calculamos el cambio local $\Delta E$ producido por el intercambio de dos espines vecinos $k$ y $k'$. Basándonos en el Hamiltoniano de interacción $H = -J \sum_{\langle ij \rangle} s_i s_j$ , la variación de energía se implementa mediante la expresión:  $$\Delta E = 2J\left[s_k^{\mu}\sum_{i \in V_k}s_i^{\mu}+s_{k^{\prime}}^{\mu}\sum_{j \in V_{k'}}s_j^{\mu}\right]$$Donde $V_k$ y $V_{k'}$ representan los vecinos de los sitios $k$ y $k'$ excluyendo la mutua posición. La aceptación de este intercambio se rige por el criterio de Metropolis:  $$P_{C\rightarrow C^{\prime}}=\min(1,e^{-\beta\Delta E})$$Este algoritmo satisface el balance detallado, asegurando que el sistema evolucione hacia el equilibrio con la distribución de Boltzmann correcta.

# Código

Se muestra la estructura que incluye la configuración, la función de paso Monte Carlo optimizada y el bucle de medición; con fines meramente informativos. En la práctica, se han implementado estos métodos dentro de un bucle que recorre las distintas temperaturas del sistema con el fin de elaborar las gráficas que posteriormente se mostrarán en la sección de resultados. El siguiente es por tanto solo un fragmento del código que pretende ilustrar el núcleo en sí del método de Kawasaki, no el código completo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

# --- 1. Definición del núcleo de simulación ---

@njit
def kawasaki_step(lattice, beta):
    """Realiza un intercambio aleatorio según Metropolis."""
    N = lattice.shape[0]
    # Elegir sitio 1
    i1, j1 = np.random.randint(1, N-1), np.random.randint(0, N)
    
    # Elegir vecino (Kawasaki solo permite intercambio de vecinos)
    # Direcciones: 0:arriba, 1:derecha, 2:abajo, 3:izquierda
    d = np.random.randint(0, 4)
    if d == 0: i2, j2 = (i1-1), j1
    elif d == 1: i2, j2 = i1, (j1+1)%N
    elif d == 2: i2, j2 = (i1+1), j1
    else: i2, j2 = i1, (j1-1)%N
    
    # Limites (Kawasaki requiere sitios activos, evitando bordes fijos y1, yN)
    if i2 < 1 or i2 > N-2: return False
    
    s1, s2 = lattice[i1, j1], lattice[i2, j2]
    if s1 == s2: return False
    
    # Calculo local de Delta E (evitando suma sobre toda la red)
    sum1 = lattice[(i1-1)%N, j1] + lattice[(i1+1)%N, j1] + lattice[i1, (j1-1)%N] + lattice[i1, (j1+1)%N] - s2
    sum2 = lattice[(i2-1)%N, j2] + lattice[(i2+1)%N, j2] + lattice[i2, (j2-1)%N] + lattice[i2, (j2+1)%N] - s1
    
    dE = 2 * (s1 * sum1 + s2 * sum2)
    
    # Criterio de Metropolis
    if dE <= 0 or np.random.rand() < np.exp(-beta * dE):
        lattice[i1, j1], lattice[i2, j2] = s2, s1
        return True
    return False

# --- 2. Preparación del sistema ---

def initialize_lattice(N, m0):
    """Inicializa red con bordes fijos en Y y magnetización m0."""
    lat = np.ones((N, N), dtype=np.int8)
    lat[0, :] = -1
    lat[-1, :] = 1
    
    # Rellenar resto aleatoriamente para cumplir m0
    active_sites = (N-2) * N
    num_up = int((active_sites * (1 + m0)) / 2)
    arr = np.array([1]*num_up + [-1]*(active_sites - num_up))
    np.random.shuffle(arr)
    lat[1:N-1, :] = arr.reshape(N-2, N)
    return lat

# --- 3. Ejecución y medidas ---
# Se implementarían, dentro de un bucle para las temperaturas deseadas y con el N deseado. Posteriormente, se extraerían
# los datos pertinentes según la propiedad que se desee estudiar en cada caso. 

# Resultados


## Resultados para magnetización inicial nula (m₀ = 0)

<div style="display:flex; gap:10px;">

<img src="resultados_ising/snapshot_N128_T3.0_m0_0.0.jpg" style="width:32%;">

<img src="resultados_ising/snapshot_N128_T2.2_m0_0.0.jpg" style="width:32%;">

<img src="resultados_ising/snapshot_N128_T1.5_m0_0.0.jpg" style="width:32%;">

</div>

<table>
<tr>
<td><img src="resultados_ising/perfil_densidad_N128_m0_0.0.jpg" width="400"></td>
<td><img src="resultados_ising/chi_m0_0.0.jpg" width="400"></td>
<td><img src="resultados_ising/cv_m0_0.0.jpg" width="400"></td>
</tr>

<tr>
<td><img src="resultados_ising/e_m0_0.0.jpg" width="400"></td>
<td><img src="resultados_ising/extrapolacion_m0_0.0.jpg" width="400"></td>
<td><img src="resultados_ising/m_m0_0.0.jpg" width="400"></td>
</tr>
</table>

En el caso simétrico el sistema reproduce el comportamiento esperado de una **transición de fase continua**, y prácticamente todas las magnitudes medidas resultan coherentes con dicho escenario.

**Fotogramas (configuraciones instantáneas).** A T = 1.5 se observa una separación de fases limpia, con el dominio +1 ocupando la mitad inferior y el dominio −1 la superior, y una interfaz nítida y casi plana con muy pocos defectos. A T ≈ 2.2, en las proximidades de la temperatura crítica, la interfaz se vuelve marcadamente rugosa y aparecen burbujas de una fase dentro de la otra a ambos lados, tal como cabe esperar cerca de T_c. A T = 3.0 la configuración es esencialmente isótropa y desordenada, sin interfaz horizontal definida, conservándose únicamente un débil gradiente vertical residual provocado por el anclaje de las filas fijas (−1 arriba, +1 abajo). La evolución de los tres fotogramas es, por tanto, la esperada.

**Magnetización por dominio.** La curva ⟨m⟩(T) decrece de forma monótona desde valores próximos a 0.95 a baja temperatura hasta valores cercanos a cero a alta temperatura, reflejando el paso de la fase separada (ordenada) al estado homogéneo mezclado. El comportamiento es el correcto, y la caída se hace algo más marcada al aumentar el tamaño del sistema, como corresponde a una transición continua.

**Energía por partícula.** Crece de forma suave y monótona con la temperatura, con el punto de inflexión (máxima pendiente) localizado en torno a T ≈ 2.3, justo en la región de la transición. Las curvas para distintos tamaños están bien ordenadas, siendo la energía más negativa a baja T para N grande, lo que es razonable porque el peso relativo de la interfaz y de los bordes disminuye con el tamaño. Es una de las magnitudes más limpias y se ajusta a lo esperado.

**Calor específico.** Presenta máximos en la región crítica que decaen a ambos lados, con la particularidad de que el pico se desplaza hacia temperaturas menores al aumentar N. Esta es precisamente la tendencia correcta de tamaño finito, y supone una clara mejora frente a versiones previas de la simulación en las que el sistema mayor no llegaba a equilibrar a alta temperatura.

**Susceptibilidad.** Muestra igualmente un crecimiento, un máximo cerca de T_c y un decaimiento posterior, sin el crecimiento monótono espurio que aparecía cuando el sistema no equilibraba. Es la magnitud más ruidosa de todas, por basarse en las fluctuaciones de la magnetización, lo que provoca que los picos individuales fluctúen bastante de un tamaño a otro.

**Perfil de densidad ρ(y).** Ilustra de forma muy visual la misma transición: a baja temperatura es un escalón abrupto que separa las dos fases, y al aumentar T la interfaz se ensancha progresivamente hasta convertirse en un perfil suave y casi lineal a T = 3.1, que pasa por ρ ≈ 0.5 en el centro. Reproduce cualitativamente el comportamiento del ensanchamiento de interfaz descrito en la bibliografía.

**Extrapolación de T_c.** La extrapolación al límite termodinámico (1/N → 0) sitúa la temperatura crítica en T_c ≈ 2.11 a partir de los picos del calor específico y en T_c ≈ 2.58 a partir de los de la susceptibilidad. Ambos valores **encierran el resultado exacto de Onsager** (T_c = 2.269) y son por tanto plenamente compatibles con él dentro del error, lo que constituye un resultado satisfactorio.

**Ruido estadístico.** La principal limitación de esta tanda es el ruido estadístico, especialmente acusado en el calor específico y la susceptibilidad por tratarse de magnitudes basadas en fluctuaciones, que convergen lentamente. Promediar sobre un mayor número de realizaciones independientes reduciría dicha dispersión y haría más robusta la localización de los picos y, con ella, la extrapolación de T_c.


## Resultados para magnetización inicial no nula (m₀ = 0.4)

<table>
<tr>
<td><img src="resultados_ising/snapshot_N128_T1.5_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/snapshot_N128_T2.2_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/snapshot_N128_T3.0_m0_0.4.jpg" width="300"></td>
</tr>
</table>


<table>
<tr>
<td><img src="resultados_ising/perfil_densidad_N128_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/chi_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/cv_m0_0.4.jpg" width="300"></td>
</tr>

<tr>
<td><img src="resultados_ising/e_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/extrapolacion_m0_0.4.jpg" width="300"></td>

<td><img src="resultados_ising/m_m0_0.4.jpg" width="300"></td>
</tr>
</table>



En el caso asimétrico el sistema separa correctamente en dominios con la proporción impuesta por la magnetización fijada, y la mayoría de las magnitudes son coherentes; el único punto que no se aprecia con la nitidez deseada es la discontinuidad de la magnetización.

**Fotogramas y separación de fases.** El perfil de densidad a T = 1.5 muestra el escalón situado en torno a la fila y ≈ 38, es decir, en el 30 % superior del sistema, de acuerdo con la fracción esperada x = (1 + m₀)/2 = 0.7 de espines +1 en la parte inferior. Los fotogramas confirman esta separación 30/70: a T = 1.5 una banda fina de fase −1 arriba y un dominio mayoritario +1 abajo con interfaz nítida; a T = 2.2 la interfaz rugosa con burbujas; y a T = 3.0 un estado mezclado dominado por la mayoría +1 con un gradiente vertical débil. La inicialización asimétrica y la separación de fases funcionan correctamente.

**Magnetización por dominio.** Una comprobación importante es su límite de alta temperatura, que tiende a ⟨m⟩ ≈ 0.4 en lugar de a cero. Esto es exactamente lo esperado: con m₀ = 0.4 conservado, en el estado mezclado homogéneo cada mitad presenta una magnetización media igual a m₀, lo que verifica además que la dinámica de Kawasaki conserva la magnetización total a lo largo de toda la simulación. Sin embargo, la **discontinuidad** propia de una transición de primer orden que cabría esperar en esta curva no se aprecia con claridad, resultando m(T) suave y sigmoidal. Esto se atribuye, por un lado, a que una transición de primer orden solo es estrictamente discontinua en el límite termodinámico y a tamaño finito (N = 128) aparece redondeada, y por otro a que la dinámica de Kawasaki equilibra muy lentamente en las proximidades de una transición de primer orden, de modo que el calentamiento progresivo (annealing) tiende a suavizar el salto. No es un fallo de la simulación sino una limitación intrínseca de tamaño y dinámica.

**Energía por partícula.** Igual que en el caso simétrico, crece de forma suave con un punto de inflexión cerca de la transición y curvas bien ordenadas por tamaño. Es coherente con lo esperado.

**Calor específico y susceptibilidad.** Ambas magnitudes presentan el mismo comportamiento cualitativo que en el caso simétrico: crecimiento, máximo en la región T ≈ 2.4–2.6 y decaimiento posterior. Son, de nuevo, las magnitudes más afectadas por el ruido estadístico, por lo que la posición exacta de sus picos baila apreciablemente entre tamaños.

**Perfil de densidad ρ(y).** Reproduce la misma física que los fotogramas: escalón abrupto al 30 % a baja T que se ensancha al aumentar la temperatura, tendiendo a alta T a un perfil cuyo valor medio es ρ ≈ 0.7, consistente con la densidad asociada a m₀ = 0.4.

**Extrapolación de T_c y ruido estadístico.** La extrapolación sitúa la temperatura en torno a T_c ≈ 2.6, con los puntos del calor específico bastante dispersos respecto a la recta de ajuste. Conviene interpretar este número con cautela: para el caso asimétrico la comparación directa con el T_c de Onsager (2.269) pierde parte de su sentido físico, al no tratarse del punto crítico del modelo de Ising simétrico, sino de la temperatura a la que se disuelve la fase separada con condiciones de contorno fijas. Como en el caso anterior, el ruido estadístico en C_v y χ es la principal limitación, y aumentar el número de realizaciones independientes mejoraría tanto la suavidad de las curvas como la fiabilidad de la extrapolación.